# DATASCI 350 - Data Science Computing

## Assignment 07 - Language models through an API

### Instructions

This assignment evaluates your understanding of lectures 14 and 15: calling a model over an API, API keys, token counts and cost, structured output, agent loops, embeddings, and retrieval-augmented generation.

Before you start, check that you have four things. First, Ollama running, with `llama3.2:1b` pulled in lecture 12; `ollama ls` should list it. Second, the embedding model for the last two tasks: run `ollama pull embeddinggemma` once. Third, the two Python packages used by the demo scripts: `pip install openai pandas ollama`. Fourth, the demo files from the course repository, which each task links to. Every task downloads its files with `curl`, which macOS ships with. On Linux, install it with `sudo apt update && sudo apt install curl -y` if it is missing. On Windows, work inside WSL or Git Bash, where `curl` is available.

Five tasks need you to run a model and report what it produced: tasks 02, 06, 07, 09 and 10. Task 04 asks you to run a few commands and read the result. Tasks 01, 03, 05 and 08 are reading and reasoning about code you already have.

No task needs an OpenRouter key, and nothing here needs an internet connection once the models are pulled and the files are downloaded. Everything runs against the server on your own laptop.

Your outputs will not match the slides, and they do not need to. Models differ between machines and between versions. Report what your machine did and paste the actual output. Do not edit outputs.

You must complete this assignment individually. You may use notes, books, and AI tools, but you must submit original work. I strongly recommend that you do not use AI tools to generate your solutions, as this will not help you learn the material. Acknowledge all resources used, including input from classmates and AI, in a short note at the end of your submission. If you are unsure about permissible resources or proper acknowledgement, please ask the instructor.

### Submission

Please submit your solutions as either a single Jupyter notebook or a PDF file. For each task, paste the commands you ran, any script you wrote or edited, the output you got, and your written answer where the task asks for one. Screenshots are welcome but not required if you paste the text. Submit your completed assignment to Canvas.

### Task 01

This is the body of `ask_local.py` from lecture 14:

```python
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # the library insists on one; Ollama ignores it
)

response = client.chat.completions.create(
    model="llama3.2:1b",
    messages=[{"role": "user", "content": "Why is the sky blue?"}],
    temperature=0,
)
```

The script talks to a model running on your own laptop. No request leaves your machine. Yet it imports a package called `openai` and passes `api_key="ollama"`.

In one paragraph, answer three things: why the package has that name even though no OpenAI service is involved, why the `api_key` argument has to be there at all, and why the string `"ollama"` works just as well as any other string would.

Hint: Appendix 01 of lecture 14, and the "The same call, from Python" slide.

*Commands, output, and answer here.*

### Task 02

Download `ask_local.py`:

```bash
curl -O https://raw.githubusercontent.com/danilofreire/datasci350/main/lectures/lecture-14/demo/ask_local.py
```

Open the file in your editor. You will change two things.

1. Above the `response = ...` block, create a list called `questions` holding three questions of your own, as strings. Pick questions of clearly different lengths.
2. Wrap the `response = ...` block and the `print` calls in a `for` loop over `questions`, so that each question is sent in its own call. Inside the loop, print the answer and then print `response.usage`.

Run it:

```bash
python3 ask_local.py
```

Paste your edited script and the full output.

Then answer in a few sentences: which of the three questions cost the most tokens in total, and was that driven by the question (`prompt_tokens`) or by the answer (`completion_tokens`)?

*Commands, output, and answer here.*

### Task 03

Here are two real responses to the same prompt, "In two sentences, what is Rayleigh scattering?", sent to `llama3.2:1b` at temperature 0.

First response:

```text
Rayleigh scattering is the scattering of light by small particles or molecules
in the atmosphere, where shorter wavelengths of light are scattered more than
longer wavelengths, resulting in a blue color of the sky during the day

finish_reason: length
```

Second response:

```text
Rayleigh scattering is the scattering of light by small particles or molecules
in the atmosphere, where shorter wavelengths of light are scattered more than
longer wavelengths, resulting in a blue color of the sky during the day. This
phenomenon is named after the British physicist Lord Rayleigh, who first
described it in the late 19th century, and is a key factor in the Earth's
apparent blue color.

finish_reason: stop
```

Answer three questions, a few sentences each.

1. One argument was passed to `chat.completions.create` in the first call and left out of the second. Name it and give the value it must have had.
2. What does `finish_reason: length` tell you, and how does that differ from `finish_reason: stop`?
3. The first call raised no error. Suppose it sat inside a loop over a thousand rows of data, writing each answer to a CSV. Why is a silent truncation worse than a crash here?

Hint: the "What comes back" slide.

*Commands, output, and answer here.*

### Task 04

You do not need a real API key for this task. You will never put one in a file here.

Make a new folder outside your course repository and create three files in it.

`.env`, with one line:

```text
OPENROUTER_API_KEY=fake-key-for-the-assignment
```

`.gitignore`, with one line:

```text
.env
```

`.env.example`, with the variable name and no value:

```text
OPENROUTER_API_KEY=
```

Then run these three commands in that folder and paste the output of each:

```bash
git init
git status
git check-ignore -v .env
```

Then answer in one paragraph: which of the three files must never be committed, and what is each of the other two for? Finish with this case. A classmate committed a real key by mistake, noticed, deleted the file, and committed again. The key no longer appears in the working folder. Is it safe? Say why, and say what they should do.

Hint: the "Where the key lives" and "Getting a key" slides.

*Commands, output, and answer here.*

### Task 05

Here are three jobs. For each one, say whether you would use a local model on your own laptop or a hosted model behind an API, and give two reasons: one about the data, and one about cost or effort. A few sentences per job.

1. Classifying 200 emails from students, which contain their names and their grades.
2. Summarising 50 press releases published on a government website.
3. Drafting one job advert for a research assistant post.

Hint: the "Why run a model on your own laptop?" slide of lecture 12 and "The receipt" slide of lecture 14.

*Commands, output, and answer here.*

### Task 06

Download the classifier and its data into the same folder:

```bash
curl -O https://raw.githubusercontent.com/danilofreire/datasci350/main/lectures/lecture-14/demo/classify.py
curl -O https://raw.githubusercontent.com/danilofreire/datasci350/main/lectures/lecture-14/demo/headlines.csv
```

Run it from that folder:

```bash
python3 classify.py
```

Paste your table and your agreement line. Mine looked like this:

```text
                                      headline human_label model_label
    Chip maker warns of a sharp drop in demand     bearish     bearish
        Retailer posts record quarterly profit     bullish     bullish
  Central bank leaves interest rates unchanged     neutral     neutral
 Airline cancels orders after fuel costs surge     bearish     bearish
    Carmaker announces plans for a new factory     bullish     bullish
      Regulator opens an inquiry into the bank     bearish     bearish
Company appoints a new chief financial officer     neutral     bullish
     Software firm beats earnings expectations     bullish     bullish
Housing starts fall for a third straight month     bearish     bearish
 The index closed almost flat on light trading     neutral     neutral
Miner cuts its dividend to fund debt repayment     bearish     bearish
          Drug trial results exceed the target     bullish     bullish
     Annual report to be published on Thursday     neutral     bearish
  Supplier reports delays at two of its plants     bearish     bearish
   Energy group raises its production forecast     bullish     bullish

Model agreed with the human on 13 of 15 headlines
```

Work from your own run, not from mine. Pick one of the headlines your model got wrong. Could you defend the model's label? Answer in one paragraph, and finish by saying what you would change first, the prompt, the model, or the human labels, and why.

Hint: the "Did it get them right?" slide.

*Commands, output, and answer here.*

### Task 07

Open `classify.py` and edit the `SYSTEM` string near the top of the file. Inside the brackets after `"sentiment" (one of`, add `"unclear"` as a fourth permitted value, and add one short sentence at the end of the string saying when the model should use it. Change nothing else, and leave `headlines.csv` as it is.

Run `python3 classify.py` once and paste the whole table and the agreement line.

Then answer in a few sentences:

1. Which headlines, if any, did the model label `"unclear"`?
2. The agreement line in Task 06 said how many of the 15 headlines the model got right. What does it say now?
3. Why can `"unclear"` never count as agreement with the labels in `headlines.csv`?

Hint: the `SYSTEM` string is built from several pieces, each in its own quotes. Keep each piece's opening and closing quotes in place, and use the other kind of quote inside it.

*Commands, output, and answer here.*

### Task 08

This is the agent loop from lecture 14:

```python
messages = [{"role": "user", "content": task}]

while True:
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOL_DESCRIPTIONS,
    )
    reply = response.choices[0].message

    if not reply.tool_calls:
        break                     # the model is finished

    call = reply.tool_calls[0]
    tool_name = call.function.name
    tool_arguments = json.loads(call.function.arguments)

    my_function = TOOLS_I_ALLOW[tool_name]
    result = my_function(tool_arguments)   # your code runs it

    messages.append(reply)
    messages.append({"role": "tool",
                     "tool_call_id": call.id,
                     "content": result})
```

In one paragraph, answer three things. Which single line is the security boundary, and why that one? If the model asks for a tool that is not in `TOOLS_I_ALLOW`, what happens, and what can the model do with that tool? And what would change if `TOOLS_I_ALLOW[tool_name]` were replaced by `eval(tool_name)`? `eval` takes a string and runs it as Python code ([documentation](https://docs.python.org/3/library/functions.html#eval)).

Hint: read the "The lethal trifecta" slide alongside "The loop, in code".

*Commands, output, and answer here.*

### Task 09

Download the RAG script, which contains the `cosine` function you need:

```bash
curl -O https://raw.githubusercontent.com/danilofreire/datasci350/main/lectures/lecture-15/demo/rag.py
```

Write a short script of your own that imports `ollama`, copies the `cosine` function from `rag.py` into your own file, and embeds these three sentences with `embeddinggemma` in a single call. Copy the function rather than importing it from `rag.py`: importing runs the whole of `rag.py`, which needs the `corpus` folder you only download in Task 10.

1. "The central bank raised interest rates"
2. "Borrowing became more expensive after the announcement"
3. "The cat slept on the windowsill"

Compute all three pairwise similarities, 1 with 2, 1 with 3, and 2 with 3, and print them to three decimal places. Paste your script and the three numbers.

Then answer in a few sentences: which pair scored highest, how many content words do those two sentences share, and what does that tell you about what the embedding is measuring?

Hint: the "Measuring close: cosine similarity" slide.

*Commands, output, and answer here.*

### Task 10

You need the whole `demo` folder from lecture 15, because `rag.py` reads a `corpus` folder of markdown files sitting next to it. Two ways to get it:

```bash
git clone --depth 1 https://github.com/danilofreire/datasci350.git
cp -r datasci350/lectures/lecture-15/demo rag-demo
cd rag-demo
```

or open <https://github.com/danilofreire/datasci350/tree/main/lectures/lecture-15/demo> in your browser and download `rag.py` and every file inside `corpus/`, keeping the same folder layout.

Then, from the folder holding `rag.py`:

```bash
ollama pull embeddinggemma
pip install ollama
```

Open `rag.py` and change the `QUESTION` variable near the top to exactly this:

```python
QUESTION = "Is AI allowed in the quizzes?"
```

Run `python3 rag.py` and paste the whole output, from the `Corpus:` line to the end of the answer.

Now search the corpus yourself, from the same folder, and paste the output:

```bash
grep -i "AI allowed" corpus/*.md
```

Then answer in one paragraph. Does any of the three retrieved chunks answer the question? Which paragraph holds the answer, and was that paragraph retrieved? Why do two of the chunks have exactly the same score? Was this a retrieval failure or a generation failure, and was the model's answer the right one given what it was shown?

Finally, change the `QUESTION` variable near the top to a question of your own about this course, and run it again. Paste the second output. In a few sentences, say whether the answer came from the retrieved chunks or whether the model said it did not know, and whether that was the right outcome for the question you picked.

Hint: the "A real run" and "Did it work?" slides.

*Commands, output, and answer here.*